In [2]:
import pandas as pd, numpy as np, joblib
from scipy import stats
import darts  # pickles contain darts objects

aqi_bins = {
    "PM25": [-np.inf, 10, 20, 25, 50, np.inf],
    "PM10": [-np.inf, 20, 35, 50, 100, np.inf],
    "NO2":  [-np.inf, 40, 100, 200, 400, np.inf],
    "O3":   [-np.inf, 80, 100, 180, 240, np.inf],
    "SO2":  [-np.inf, 100, 200, 350, 500, np.inf],
}
pollutants = list(aqi_bins)

# raw daily data (unfilled), only used to mask interpolated days, same as your notebook
df = pd.read_parquet("gafanha.parquet")
df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.round("h")
daily = (df.pivot_table(index="timestamp", columns="collection", values="value", aggfunc="median")
           .asfreq("h").loc[:"2023-12-31 23:00"].drop(columns=["Radiation", "Precipitation"])
           .resample("D").median())
daily.index = daily.index.tz_localize(None)

def aqi_from(d):
    sub = pd.concat([pd.cut(d[p], bins=b, labels=[1, 2, 3, 4, 5]).astype(float) for p, b in aqi_bins.items()], axis=1)
    return sub.where(daily[pollutants].reindex(d.index).notna()).max(axis=1, skipna=True)

def pair(r):
    p = r["test_pred"].to_dataframe()
    return r["y_test"].to_dataframe().loc[p.index], p

def aqi_pair(r):
    a, p = pair(r)
    return aqi_from(a), aqi_from(p)

In [3]:
# %% Cell 2: Diebold-Mariano vs Linear Regression (h=1, squared error, Harvey-Leybourne-Newbold correction)
files = {"LR": "results_lr_multi.pkl", "VARIMA": "results_varima.pkl",
         "RF": "results_rf_multi.pkl", "XGB": "results_xgb_multi.pkl"}
R = {m: joblib.load(f) for m, f in files.items()}

# DM test for 1-day forecast only
def dm(e_base, e_model, h=1):
    d = (e_base**2 - e_model**2).dropna()          # d > 0 => model beats LR
    n = len(d)
    stat = d.mean() / np.sqrt(d.var(ddof=0) / n)  
    stat *= np.sqrt((n + 1 - 2*h + h*(h-1)/n) / n) # HLN small-sample fix
    return stat, 2 * stats.t.sf(abs(stat), n - 1)

def errs(r, target):                               # error series on the test window
    if target == "AQI":
        a, p = aqi_pair(r); return p - a
    a, p = pair(r); return p[target] - a[target]

rows = []
for period in ["pre", "during", "post"]:
    for target in ["AQI"] + pollutants:
        base = errs(R["LR"][period], target)
        for m in R:
            if m == "LR": continue
            e = errs(R[m][period], target)
            b, e = base.align(e, join="inner")     # same test days only
            s, p = dm(b, e)
            rows.append(dict(period=period, target=target, model=m,
                             rmse_LR=np.sqrt((b**2).mean()), rmse_model=np.sqrt((e**2).mean()),
                             gain_pct=100 * (1 - np.sqrt((e**2).mean() / (b**2).mean())),
                             DM=s, p=p))
dm_table = pd.DataFrame(rows)
print(dm_table.round(3).to_string())
dm_table.to_csv("dm_vs_lr.csv", index=False)

# self-check: a model with half the error must beat the baseline significantly
_r = np.random.default_rng(0); _e = pd.Series(_r.normal(size=200))
assert dm(_e, _e * 0.5)[0] > 0 and dm(_e, _e * 0.5)[1] < 0.05

    period target   model  rmse_LR  rmse_model  gain_pct      DM      p
0      pre    AQI  VARIMA    0.869       0.853     1.793   0.404  0.686
1      pre    AQI      RF    0.869       0.869     0.000   0.000  1.000
2      pre    AQI     XGB    0.869       0.871    -0.253  -0.064  0.949
3      pre   PM25  VARIMA    7.608       7.803    -2.561  -0.640  0.522
4      pre   PM25      RF    7.608       7.778    -2.238  -0.563  0.574
5      pre   PM25     XGB    7.608       7.710    -1.339  -0.366  0.715
6      pre   PM10  VARIMA   10.457       9.880     5.520   1.945  0.053
7      pre   PM10      RF   10.457      10.598    -1.349  -0.502  0.616
8      pre   PM10     XGB   10.457      10.225     2.219   0.693  0.489
9      pre    NO2  VARIMA    4.633       4.813    -3.894  -0.957  0.340
10     pre    NO2      RF    4.633       4.560     1.567   0.379  0.705
11     pre    NO2     XGB    4.633       4.476     3.388   0.998  0.319
12     pre     O3  VARIMA   14.323      13.851     3.291   1.495